In [3]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import mahalanobis

In [5]:
df = pd.read_csv('homework_8.2.csv')
df.drop(columns='Unnamed: 0', inplace=True)
df

,X,Y,Z1,Z2
0,1,4.652085,1.764052,2.320015
1,1,2.590221,0.400157,1.292631
2,1,3.898981,0.978738,0.556423
3,1,5.857179,2.240893,2.345607
4,1,3.647489,1.867558,2.095611
...,...,...,...,...
995,1,5.336848,0.412871,0.510622
996,1,0.936869,-0.198399,1.203125
997,0,-1.346745,0.094192,0.252626
998,0,0.414432,-1.147611,-2.289512


### Problem 3-4

Question 3
Using homework_8.2.csv, match all treated items to the single nearest untreated item using the Mahalanobis distance. (Do this with replacement — the same untreated item can be used again.) 

* Use the mahalanobis function from the scipy.spatial.distance module.

* For the inverse covariance matrix, use all Z=1 values and all Z=2 values, make them into a 2XN matrix, find its 2x2 covariance, and invert. 

Then, the ATE is closest to: 

In [30]:
# For the inverse covariance matrix, use all Z=1 values and all Z=2 values, make them into a 2XN matrix, find its 2x2 covariance, and invert. 
z_cols = ['Z1', 'Z2']
zs = df[z_cols]
cov_matrix = zs.cov()
inv_cov_matrix = np.linalg.inv(cov_matrix)

In [57]:
# match all treated items to the single nearest untreated item using the Mahalanobis distance.

# It was harder to work with dataframe than numpy arr for iterating
df_x1 = df[df['X'] == 1]
df_x0 = df[df['X'] == 0]
x1_zs = df_x1[z_cols].values
x0_zs = df_x0[z_cols].values

results = []
q4_results = []
for x1_row in x1_zs:
    mahalanobis_distances = [
        mahalanobis(x1_row, x0_row, inv_cov_matrix) for x0_row in x0_zs
    ]

    # "single nearest untreated item" is the smallest distance
    best_x0_index = np.argmin(mahalanobis_distances)

    # For problem 4 we also need the farthest distance, so we need to persist the distance as well as its index
    best_x0_distance = np.min(mahalanobis_distances)
    q4_results.append(best_x0_distance)

    # Map back to the original untreated dataframe index
    results.append(df_x0.index[best_x0_index])
results

df_x1_x0_matched = df_x1.copy()
df_x1_x0_matched['matched_x0_index'] = results
df_x1_x0_matched

,X,Y,Z1,Z2,matched_x0_index
0,1,4.652085,1.764052,2.320015,681
1,1,2.590221,0.400157,1.292631,752
2,1,3.898981,0.978738,0.556423,892
3,1,5.857179,2.240893,2.345607,922
4,1,3.647489,1.867558,2.095611,922
...,...,...,...,...,...
992,1,1.763565,-0.799422,0.515715,910
993,1,3.871122,0.240788,-0.082670,888
994,1,1.394735,0.289121,0.486949,18
995,1,5.336848,0.412871,0.510622,381


In [58]:
merged_df = pd.merge(df_x1_x0_matched, df_x0, left_on="matched_x0_index", right_index=True, suffixes=("_x1", "_x0"))
merged_df

,X_x1,Y_x1,Z1_x1,Z2_x1,matched_x0_index,X_x0,Y_x0,Z1_x0,Z2_x0
0,1,4.652085,1.764052,2.320015,681,0,0.428954,1.124419,1.663580
1,1,2.590221,0.400157,1.292631,752,0,-0.034844,0.448195,1.382375
2,1,3.898981,0.978738,0.556423,892,0,1.164988,0.930408,0.573537
3,1,5.857179,2.240893,2.345607,922,0,1.797450,1.325014,1.169978
4,1,3.647489,1.867558,2.095611,922,0,1.797450,1.325014,1.169978
...,...,...,...,...,...,...,...,...,...
992,1,1.763565,-0.799422,0.515715,910,0,-2.545817,-0.817493,0.440251
993,1,3.871122,0.240788,-0.082670,888,0,1.191166,0.271170,-0.126742
994,1,1.394735,0.289121,0.486949,18,0,-0.426837,0.313068,0.528033
995,1,5.336848,0.412871,0.510622,381,0,0.256826,0.387280,0.551924


In [59]:
# Now calculate the effect
ate = (merged_df['Y_x1'] - merged_df['Y_x0']).mean()
ate

np.float64(3.4376789979126094)

It is the closest to 3.4 (Option C)

Question 4. Find the nearest Z1 and Z2 values of the treated item with the least common support (the farthrest Mahalanobis distance from the untreated)

In [65]:
farthest_mahalanobis_distance_index = np.argmax(q4_results)
farthest_mahalanobis_distance_index # argmax will give us the index of the row in the merged df with the largest distance

np.int64(241)

In [66]:
merged_df.iloc[farthest_mahalanobis_distance_index] # lookup the index returned by argmax

X_x1                  1.000000
Y_x1                  6.540236
Z1_x1                 2.696224
Z2_x1                 0.538155
matched_x0_index    418.000000
X_x0                  0.000000
Y_x0                  5.407995
Z1_x0                 1.519995
Z2_x0                -1.282208
Name: 494, dtype: float64

The treated item Zs are 1.52 and -1.28. closest is A (1.5 and -1.3)